-------------------------------------------
# Visualistaion géographique des villes pondéres par les flux (négatif en orange, positif en bleu). Plus le solde est grand, plus le node doit être grand aussi. 


# Spatial networks  
Associer en termes de position et la porba d'observer un edge depend de la distance. 
Tester plusieurs distances - EUclédienne, great-circle ? 

# Modèle gravitire ! 
Ou le radiqation model ?
Voir package pytdlm 


In [6]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# 1. Chargement avec conversion explicite en STR pour les codes INSEE
df_flux = pd.read_csv('BDD/edges_mobilites_nb_code_insee.csv', dtype={'source': str, 'target': str})
df_coords = pd.read_csv("BDD/communes.csv", dtype={'code_insee': str})

# 2. Calcul du solde net par ville
# Somme des arrivées (target)
arrivals = df_flux.groupby('target')['nb_migrations'].sum().reset_index()
# Somme des départs (source)
departures = df_flux.groupby('source')['nb_migrations'].sum().reset_index()

# Fusion des flux entrants et sortants
villes_stats = pd.merge(
    arrivals, 
    departures, 
    left_on='target', 
    right_on='source', 
    how='outer'
).fillna(0)

# Création de la colonne de clé unique (Code INSEE)
villes_stats['code_insee'] = villes_stats['target'].where(villes_stats['target'] != 0, villes_stats['source'])

# Calcul du solde net et du flux total (volume)
villes_stats['flux_net'] = villes_stats['nb_migrations_x'] - villes_stats['nb_migrations_y']
villes_stats['total_flux'] = villes_stats['nb_migrations_x'] + villes_stats['nb_migrations_y']

# 3. Ajout des coordonnées et populations
villes_stats = villes_stats.merge(df_coords, on='code_insee', how='left')

# Nettoyage des données sans coordonnées
villes_stats = villes_stats.dropna(subset=['latitude', 'longitude'])

# Définition des couleurs selon le solde (Orange pour perte, Bleu pour gain)
villes_stats['couleur_solde'] = villes_stats['flux_net'].apply(lambda x: 'Perte (Orange)' if x < 0 else 'Gain (Bleu)')

# --- VISUALISATION 1 : CARTE GLOBALE ---
fig1 = px.scatter_mapbox(
    villes_stats, 
    lat="latitude", lon="longitude", 
    size=villes_stats['total_flux'].abs(), # Taille basée sur le volume total
    color="couleur_solde",
    color_discrete_map={"Perte (Orange)": "orange", "Gain (Bleu)": "blue"},
    hover_name="code_insee",
    hover_data=['flux_net', 'population'],
    title="Flux migratoires par ville (Taille = Volume total, Couleur = Solde Net)",
    mapbox_style="carto-positron", zoom=5, center={"lat": 46.2, "lon": 2.2}
)
fig1.show()

# --- VISUALISATION 2 : TOP 10 ET BOTTOM 10 (BAR CHART) ---
top_10 = villes_stats.nlargest(10, 'flux_net')
bottom_10 = villes_stats.nsmallest(10, 'flux_net')
df_extremes = pd.concat([top_10, bottom_10]).sort_values('flux_net', ascending=False)

fig2 = px.bar(
    df_extremes,
    x='code_insee', y='flux_net',
    color='couleur_solde',
    color_discrete_map={"Perte (Orange)": "orange", "Gain (Bleu)": "blue"},
    title="Top 10 Gains vs Top 10 Pertes Migratoires",
    labels={'flux_net': 'Solde Net', 'code_insee': 'Code INSEE'}
)
fig2.show()

# --- VISUALISATION 3 : CARTE DES EXTRÊMES ---
fig3 = px.scatter_mapbox(
    df_extremes, 
    lat="latitude", lon="longitude", 
    size=df_extremes['flux_net'].abs(), 
    color="couleur_solde",
    color_discrete_map={"Perte (Orange)": "orange", "Gain (Bleu)": "blue"},
    hover_name="code_insee",
    title="Localisation des 20 villes avec les plus gros écarts de flux",
    mapbox_style="carto-positron", zoom=5
)
fig3.show()

C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_15140\2826913339.py:41: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig1 = px.scatter_mapbox(


C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_15140\2826913339.py:70: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig3 = px.scatter_mapbox(


In [ ]:
# --- ÉTAPE 2 : Top 10 Arrivées vs Départs ---

# Top 10 (Plus gros solde positif)
top_10 = villes_stats.nlargest(10, 'flux_net')
# Bottom 10 (Plus gros solde négatif / Pertes)
bottom_10 = villes_stats.nsmallest(10, 'flux_net')

combined_top_bottom = pd.concat([top_10, bottom_10])

# Visualisation en Bar Chart
fig2 = px.bar(
    combined_top_bottom, 
    x='ville', y='flux_net',
    color='couleur',
    title="Top 10 et Bottom 10 des villes par solde migratoire",
    labels={'flux_net': 'Solde Migratoire Net', 'ville': 'Ville'}
)
fig2.show()

# Visualisation sur Carte filtrée
fig3 = px.scatter_mapbox(
    combined_top_bottom, 
    lat="latitude", lon="longitude", 
    size=combined_top_bottom['flux_net'].abs(), # On prend la valeur absolue pour la taille
    color="couleur",
    hover_name="ville",
    title="Localisation des 20 villes les plus dynamiques (Gains/Pertes)",
    mapbox_style="carto-positron", zoom=5
)
fig3.show()

In [ ]:
import numpy as np
import pandas as pd

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calcule la distance en km entre deux points GPS."""
    R = 6371  # Rayon de la Terre en km
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

def modele_gravitaire(pop_source, pop_target, distance, k=1, beta=2):
    """Applique la formule gravitaire."""
    if distance == 0: return 0
    return k * (pop_source * pop_target) / (distance**beta)

# --- Application sur ton DataFrame ---

# On suppose que tu as fusionné tes flux avec les données villes_coords
# Colonnes requises : lat_s, lon_s, pop_s (source) et lat_t, lon_t, pop_t (target)

# 1. Calcul de la distance pour chaque ligne
df_flux['distance'] = df_flux.apply(
    lambda row: haversine_distance(row['lat_s'], row['lon_s'], row['lat_t'], row['lon_t']), 
    axis=1
)

# 2. Calcul du flux théorique (Modèle Gravitaire)
# Ajuste k et beta selon tes observations réelles
df_flux['flux_theorique'] = df_flux.apply(
    lambda row: modele_gravitaire(row['pop_s'], row['pop_t'], row['distance'], k=0.001, beta=1.5),
    axis=1
)

# 3. Comparaison : On calcule l'écart entre le réel et le modèle
df_flux['residuel'] = df_flux['nb_migrations'] - df_flux['flux_theorique']

In [ ]:
import numpy as np
import pandas as pd

def compute_radiation_flux(df_flux, df_villes):
    """
    df_villes: DataFrame avec [ville_id, pop, lat, lon]
    df_flux: DataFrame avec [source, target, flux_reel]
    """
    
    # 1. Pré-calculer le total des départs réels par ville (Ti)
    total_departures = df_flux.groupby('source')['flux_reel'].sum().to_dict()
    
    # 2. Fonction pour calculer s_ij (population intermédiaire)
    def get_s_ij(row, all_villes):
        # Distance entre source et cible
        d_limit = row['distance']
        source_id = row['source']
        target_id = row['target']
        
        # On filtre les villes qui sont plus proches de la source que la cible
        # mais qui ne sont ni la source ni la cible
        mask = (all_villes['dist_to_source'] < d_limit) & \
               (all_villes['ville_id'] != source_id) & \
               (all_villes['ville_id'] != target_id)
        
        return all_villes.loc[mask, 'pop'].sum()

    # 3. Calcul du flux théorique
    results = []
    for source in df_villes['ville_id'].unique():
        # On calcule les distances de toutes les autres villes par rapport à cette source
        # (Utilise une fonction Haversine comme vu précédemment)
        villes_copy = df_villes.copy()
        villes_copy['dist_to_source'] = calculate_distances(source, villes_copy)
        
        Ti = total_departures.get(source, 0)
        mi = df_villes.loc[df_villes['ville_id'] == source, 'pop'].values[0]
        
        # Pour chaque destination j
        for _, dest_row in villes_copy.iterrows():
            if dest_row['ville_id'] == source: continue
            
            nj = dest_row['pop']
            rij = dest_row['dist_to_source']
            
            # Population dans le cercle
            s_ij = villes_copy[(villes_copy['dist_to_source'] < rij) & 
                               (villes_copy['ville_id'] != source) & 
                               (villes_copy['ville_id'] != dest_row['ville_id'])]['pop'].sum()
            
            # Formule de Radiation
            denom = (mi + s_ij) * (mi + nj + s_ij)
            flux_predit = Ti * (mi * nj) / denom
            
            results.append({
                'source': source, 
                'target': dest_row['ville_id'], 
                'flux_radiation': flux_predit
            })
            
    return pd.DataFrame(results)